In [0]:
dbutils.widgets.text("p_batch_id", "", "Batch ID")

v_batch_id = dbutils.widgets.get("p_batch_id").strip()

if not v_batch_id:
    raise ValueError("Enter a batch ID")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/04.gold-helpers

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
races_silver_path = "s3://bizf1project123/Silver/Aaisha/incremental-load/races"

circuits_silver_path = "s3://bizf1project123/Silver/Aaisha/incremental-load/circuits"

gold_path = "s3://bizf1project123/Gold/Aaisha/incremental-load/dim_races"

In [0]:
circuits_df = (
    spark.read.format("delta")
        .load("s3://bizf1project123/Silver/Aaisha/incremental-load/circuits")
)

In [0]:
races_df = (
    spark.read.format("delta")
        .load(races_silver_path)
        .filter(F.col("batch_id") == v_batch_id)
)

In [0]:
display(races_df)

In [0]:
display(circuits_df)

In [0]:
dim_races_df = (
    races_df.alias("r")
        .join(
            circuits_df.alias("c"),
            F.col("r.circuit_id") == F.col("c.circuit_id"),
            "inner"
        )
        .select(
            F.col("r.season"),
            F.col("r.round"),
            F.col("r.race_name"),
            F.col("r.race_date"),
            F.col("c.circuit_name"),
            F.col("c.locality"),
            F.col("c.country"),
            F.col("r.batch_id")
        )
)

In [0]:
display(dim_races_df)

In [0]:
dim_races_df = (
    dim_races_df
        .withColumn("created_timestamp", F.current_timestamp())
        .withColumn("updated_timestamp", F.current_timestamp())
)

In [0]:
display(dim_races_df)

In [0]:
if not DeltaTable.isDeltaTable(spark, gold_path):
    (
        dim_races_df.write
            .format("delta")
            .mode("overwrite")
            .save(gold_path)
    )
else:
    gold_delta = DeltaTable.forPath(spark, gold_path)

    (
        gold_delta.alias("t")
            .merge(
                dim_races_df.alias("s"),
                "t.season = s.season AND t.round = s.round"
            )
            .whenMatchedUpdate(
                set={
                    "race_name": "s.race_name",
                    "race_date": "s.race_date",
                    "circuit_name": "s.circuit_name",
                    "locality": "s.locality",
                    "country": "s.country",
                    "updated_timestamp": "s.updated_timestamp"
                }
            )
            .whenNotMatchedInsertAll()
            .execute()
    )

In [0]:
dim_races_gold_df = spark.read.format("delta").load(gold_path)

display(dim_races_gold_df)